# OCR do Livro de Registro de Leis nº 1 (1892–1894): resultados

Este caderno mostra o pipeline final aplicado às **12 páginas** transcritas (páginas 5 a 16). Ele não refaz nenhum experimento: a configuração final já foi decidida e está em `configs/final.json`. Para cada página, mostra:

1. cada operação de imagem aplicada **sobre a original**, em sequência, e a segmentação de linhas desenhada sobre a imagem;
2. cada etapa de texto (reconhecedor → reescore CTC → convenções → hifenização → correção léxica) com o **WER contra o GT**;
3. o texto final ao lado do GT, com as palavras erradas marcadas, e o recorte de cada linha ao lado do texto lido dela.

O processo completo está em `PROCESSO.md`.

## Sem vazamento de dados

- **O reconhecedor é o PP-OCRv6 medium, como foi publicado.** Nenhum peso foi treinado com estas páginas.
- **O pós-processamento não usa nenhuma transcrição deste livro.** O léxico é genérico (pt-BR 50k, legendas de filmes) e as regras de notação são fixas.
- **Só parâmetros foram ajustados**, como limiares, operações de imagem e pesos do léxico, e só nas **9 páginas de ajuste** (5, 6, 8, 9, 11, 13, 14, 15, 16).
- **As páginas 7, 10 e 12 ficaram reservadas** e não participaram de nenhuma escolha. O número delas é a estimativa honesta. O das páginas de ajuste é um pouco otimista, porque os parâmetros foram escolhidos olhando para elas.

## Resultado

WER normalizado (minúsculas, sem pontuação), somando as edições do corpus. Quanto menor, melhor.

| configuração | ajuste (9) | **reservadas (7, 10, 12)** | todas (12) |
|---|---:|---:|---:|
| modelo como publicado, imagem crua, sem pós-processamento | 63,15 | 64,57 | 63,52 |
| **final** (`configs/final.json`) | 46,95 | **48,66** | 47,40 |

A meta do projeto era WER normalizado ≤ 50%. Nas reservadas, o CER normalizado cai de 21,71% para 17,79%.

## O que foi testado e o que ficou

### Arquitetura

Três fluxos de imagem compartilham a mesma geometria:

```
original → geometria (ampliação, inclinação)  = base (define as coordenadas)
base → operações da segmentação   → segmentador neural blla (Kraken) → linhas (polígonos)
base → operações do reconhecimento → PP-OCRv6 medium (Kraken) → texto → pós-processamento
```

As operações fotométricas não movem pixels, então os polígonos achados numa imagem valem na outra. Isso permite **segmentar numa imagem e reconhecer em outra**. O primeiro ganho veio daí: binarizar ajudava a segmentação, mas o reconhecedor perdia 9,6 pontos lendo pixels binarizados.

### Progressão nas páginas de ajuste (WER norm. %)

| etapa | mudança | WER |
|---|---|---:|
| baseline antiga | binarização nlbin na segmentação **e** no reconhecimento | 70,54 |
| crua | sem tratamento | 63,15 |
| fluxos separados | segmenta na imagem binarizada, reconhece na original | 60,95 |
| + convenções e hifenização | `n^o→nº`, `⁋→§`, junta palavra partida por `-` | 59,20 |
| + correção pelo léxico pt-BR | troca palavras fora do léxico pela vizinha mais próxima | 56,77 |
| busca gulosa condicional (final) | todas as etapas redecididas em conjunto, com reescore CTC | **46,95** |

### Como a configuração final foi encontrada: busca gulosa condicional

Testar cada mudança isolada sobre a crua e somar as que ganham não funciona, porque os efeitos interagem. A busca roda só nas 9 páginas de ajuste e segue estas regras:

- As etapas são decididas em ordem, sempre **sobre o melhor conjunto atual**.
- Em cada etapa, cada candidato é testado sozinho. O melhor entra só se superar a referência.
- Depois, **todos os outros** candidatos da etapa são testados por cima do vencedor, inclusive os que pioraram sozinhos. Cada um fica só se melhorar.
- Um passe para trás remove qualquer escolha cuja retirada não piore.
- Os passes se repetem até nada mudar. Nenhuma decisão é definitiva.

Foram 3 passes e 330 configurações avaliadas.

| etapa | passe 1: escolha | passe 2 em diante: escolha |
|---|---|---|
| reconhecedor | medium (small piora) | medium |
| imagem de segmentação | nlbin + contraste | **iluminação + CLAHE** |
| geometria | ampliação 1,5 | ampliação 1,5 + inclinação |
| filtro de linhas | linhas < 5% da largura | nenhum |
| imagem de reconhecimento | iluminação + nitidez + gama 1,5 | iluminação + gama 0,7 |
| texto | convenções + hifenização | igual |
| léxico | pt-BR 50k inteiro, razão 0,5, tamanho mínimo 3 | igual |
| decodificador | ctc+edit, α = 16 | **α = 2** |

O que a busca mostrou:

- **O contraste piorou sozinho (+0,73) e ajudou sobre o nlbin.** Somando só o que melhora isoladamente, essa combinação nunca seria testada.
- **No passe 2 a segmentação trocou nlbin por iluminação + CLAHE**, depois que o reconhecimento e o léxico estavam ativos. O nlbin, base das versões anteriores, saiu.
- Tratamentos da imagem de reconhecimento, que isoladamente ficavam no ruído, passaram a ajudar quando decididos em conjunto.
- Nas bordas da grade, um parâmetro por vez, nada passou do ruído. Testados: α 1 e 4, razão 0,45 e 0,6, gama 0,5 e 0,85, ampliação 1,25 e 1,75, e 20 candidatos CTC.

**Quanto cada escolha vale** (WER nas páginas de ajuste ao retirar só ela, medido durante a busca):

| retirar | custo |
|---|---:|
| léxico pt-BR | +8,35 |
| iluminação na imagem de reconhecimento | +5,08 |
| ampliação 1,5 | +2,03 |
| reescore ctc+edit | +2,03 |
| hifenização | +1,41 |
| léxico inteiro (50k, e não só as 20k mais frequentes) | +1,19 |
| CLAHE na segmentação | +0,73 |
| razão de edição 0,5 | +0,68 |
| convenções | +0,56 |
| inclinação | +0,51 |
| gama 0,7 | +0,51 |
| iluminação na segmentação | +0,11 |

**Retirado por precaução.** A busca também usou um léxico montado com o GT das outras páginas de ajuste, que nunca incluía a página avaliada. Mesmo assim são transcrições do conjunto que está sendo avaliado, então ele **saiu da configuração final**. Isso custa cerca de 4 pontos (43,94 → 48,66 nas reservadas).

### Outros reconhecedores, como publicados

| reconhecedor | resultado |
|---|---|
| **PP-OCRv6 medium** | **CER norm. 22,0** (páginas 6, 8, 13) |
| PP-OCRv6 small | ≈ 4 pontos de WER pior em todas as páginas |
| TraPrInq (Inquisição portuguesa, séc. XVI–XIX) | CER norm. 38,8 |
| McCATMuS (multilíngue, séc. XVI–XXI) | CER norm. 51,2 |

Não existe modelo público para manuscrito brasileiro do século XIX. Os modelos históricos portugueses disponíveis leem esta mão pior que o PP-OCRv6.

### Limites

- Só 3 páginas reservadas, ~640 palavras: 1 ponto de WER ≈ 6 palavras.
- As reservadas já tinham sido olhadas em campanhas exploratórias anteriores do projeto, antes deste pipeline. Nenhuma escolha deste pipeline foi feita com elas, mas a separação não é perfeitamente cega.
- O GT não foi verificado e tem erros prováveis (p. ex. `com causa` contra `sem causa`). Parte do erro restante é da referência.
- A página 5 é outro recorte (521×543 px, 45 palavras), e a segmentação dela falha.

## Configuração final (`configs/final.json`)

| etapa | valor |
|---|---|
| geometria | ampliação ×1,5 → correção de inclinação (±5°, passo 0,25°) |
| imagem de segmentação | normalização de iluminação (σ=25) → CLAHE (clip 2,0, grade 8) |
| segmentador | blla (padrão do Kraken), sem filtro de linhas |
| imagem de reconhecimento | normalização de iluminação (σ=25) → gama 0,7 |
| reconhecedor | PP-OCRv6 medium como publicado (Zenodo 10.5281/zenodo.21788410), padding 16 |
| reescore CTC | α = 2, 10 candidatos do léxico por palavra |
| convenções e hifenização | sim |
| léxico | só o genérico pt-BR 50k (FrequencyWords, OpenSubtitles 2018), inteiro |
| correção por edição | razão 0,5, palavras com ≥ 3 letras |

## O que cada etapa faz (e por que nenhuma usa a resposta)

**Imagem**

| etapa | o que faz |
|---|---|
| ampliação ×1,5 | aumenta a imagem; o traço fino fica com mais pixels para o segmentador |
| correção de inclinação | gira a página até as linhas ficarem horizontais |
| normalização de iluminação | divide a imagem pelo próprio fundo desfocado, tirando manchas e sombras do papel |
| CLAHE | realça o contraste local, só na imagem usada para achar as linhas |
| gama 0,7 | clareia os tons médios, só na imagem lida pelo reconhecedor |

**Texto**

| etapa | o que faz | usa o GT? |
|---|---|---|
| **reconhecedor (guloso)** | O PP-OCRv6 dá, para cada fatia da linha, a probabilidade de cada caractere. "Guloso" é pegar sempre o mais provável e juntar: é a saída crua do modelo. | não |
| **reescore CTC** | Para cada palavra lida que não está no dicionário pt-BR genérico, busca palavras parecidas no dicionário e pergunta ao próprio modelo qual delas é mais compatível com a imagem. Usa só o dicionário genérico e as probabilidades do modelo. | não |
| **convenções** | Regras fixas de notação: o modelo escreve `n^o` e `⁋`, a transcrição usa `nº` e `§`. | não usa o texto; ver a primeira ressalva |
| **hifenização** | Se a linha termina em `-`, junta com a primeira palavra da linha seguinte (`Camar-` + `go` → `Camargo`). | não |
| **correção léxica** | Troca palavras fora do dicionário genérico pela palavra mais próxima, se a diferença for pequena (até metade das letras). É um corretor ortográfico comum. | não |

**Ressalvas**

1. **Regra das convenções.** Ela existe porque foi observado como o GT representa esses símbolos. Não é a resposta de nenhuma página, e sim o padrão de notação da transcrição (como saber que o gabarito escreve § e não ⁋). Vale cerca de 0,5 ponto.
2. **Parâmetros.** O quanto corrigir (razão 0,5), o peso do reescore (α = 2) e o tratamento de imagem foram escolhidos olhando o WER das 9 páginas de ajuste. É ajuste de limiares, não de pesos do modelo. A prova de que não vale só para essas páginas: nas 3 reservadas, que não participaram de nada, o WER cai de 64,57 para 48,66, quase o mesmo ganho das páginas de ajuste (63,15 para 46,95).
3. **Generalização.** Funciona para outras páginas deste livro. Em outros livros, o pós-processamento (dicionário, hifenização, reescore) continua valendo. O tratamento de imagem (iluminação, gama, ampliação) foi escolhido para este papel e esta tinta e talvez precise ser reajustado.

## Preparação

In [ ]:
# Ambiente. No Colab: obtém o código, entra na pasta e instala as dependências (o torch do Colab é mantido).
# Localmente, abra o caderno de dentro da pasta entrega_ocr; esta célula não faz nada.
import os
import subprocess
import sys
from pathlib import Path

ORIGEM = 'github'  # 'github': clona o repositório público | 'drive': usa a pasta Meu Drive/entrega_ocr
REPOSITORIO = 'https://github.com/fabiolobato/hackathon_ocr_historico.git'
RAMO = 'feat/visual_report_greedy_search'

if 'google.colab' in sys.modules:
    if ORIGEM == 'drive':
        from google.colab import drive
        drive.mount('/content/drive')
        pasta = Path('/content/drive/MyDrive/entrega_ocr')  # ajuste se a pasta estiver em outro lugar
    else:
        clone = Path('/content/hackathon_ocr_historico')
        if not clone.is_dir():
            subprocess.run(['git', 'clone', '--depth', '1', '-b', RAMO, REPOSITORIO, str(clone)], check=True)
        pasta = clone / 'entrega_ocr'
    os.chdir(pasta)
    subprocess.run([sys.executable, 'setup_pipeline.py'], check=True)  # instala o Kraken, baixa e confere modelo e léxico
print('pasta de trabalho:', Path.cwd())

In [ ]:
import logging
import warnings

import torch
from IPython.display import HTML, display

from ocr_pipeline.config import load_config
from ocr_pipeline.data import discover, split
from ocr_pipeline.engine import Engine
from ocr_pipeline.presentation import page_html, stage_corpus, style, summary_html
from ocr_pipeline.visualize import bar_svg

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings('ignore', module='coremltools')  # aviso inofensivo do Kraken no macOS
GT = Path('ground_truth') if Path('ground_truth').is_dir() else Path('test_kraken/ground_truth')  # pasta de entrega ou repositório
PAGINAS = discover(Path('imagens'), GT)
AJUSTE, RESERVADAS = split(PAGINAS)  # semente 42: reservadas = 7, 10, 12
RESERVADAS_N = {p.number for p in RESERVADAS}
NOME = 'final'
CONFIG = load_config(Path('configs/final.json'))
DISPOSITIVO = 'cuda:0' if torch.cuda.is_available() else 'cpu'
ENGINE = Engine(Path('cache'), DISPOSITIVO, seed=42)
RESULTADOS = {NOME: {}}


def mostrar_html(corpo: str) -> None:
    display(HTML(f'{style()}<div class="ocr-body">{corpo}</div>'))


def mostrar(numero: int) -> None:
    """Roda a configuração final na página e mostra imagens, etapas de texto e WER."""
    pagina = next(p for p in PAGINAS if p.number == numero)
    tipo = 'reservada' if numero in RESERVADAS_N else 'ajuste'
    corpo, etapas = page_html(ENGINE, pagina, CONFIG, AJUSTE, f'página {numero} ({tipo})')
    RESULTADOS[NOME][numero] = etapas
    mostrar_html(corpo)


print('páginas:', [p.number for p in PAGINAS])
print('ajuste:', [p.number for p in AJUSTE], '| reservadas:', sorted(RESERVADAS_N))
print('dispositivo:', DISPOSITIVO)

## Página por página

Cada célula roda a configuração final em uma página. A primeira execução leva de 20 a 60 s por página em CPU; os resultados ficam em `cache/` e as execuções seguintes são imediatas.

Como ler cada cartão:

- **geometria / imagem de segmentação / imagem de reconhecimento**: cada imagem é a anterior com mais uma operação, e a legenda dá o valor do parâmetro. A última imagem da segmentação mostra as linhas encontradas (polígono colorido + linha de base, numeradas na ordem de leitura).
- **texto: cada etapa**: WER e CER depois de cada operação de texto, e Δ em relação à etapa anterior.
- **texto final contra o GT**: <span style="text-decoration:line-through">riscado</span> = palavra errada, em verde = o que o GT diz. Ao lado está o GT.
- **linha a linha** (expandir): recorte da imagem de reconhecimento ↔ texto lido ↔ mudanças do reescore CTC.

### Página 5 (ajuste)

In [ ]:
mostrar(5)

### Página 6 (ajuste)

In [ ]:
mostrar(6)

### Página 7 (reservada)

In [ ]:
mostrar(7)

### Página 8 (ajuste)

In [ ]:
mostrar(8)

### Página 9 (ajuste)

In [ ]:
mostrar(9)

### Página 10 (reservada)

In [ ]:
mostrar(10)

### Página 11 (ajuste)

In [ ]:
mostrar(11)

### Página 12 (reservada)

In [ ]:
mostrar(12)

### Página 13 (ajuste)

In [ ]:
mostrar(13)

### Página 14 (ajuste)

In [ ]:
mostrar(14)

### Página 15 (ajuste)

In [ ]:
mostrar(15)

### Página 16 (ajuste)

In [ ]:
mostrar(16)

## Resumo das 12 páginas

WER normalizado final por página e do corpus. Os gráficos mostram o WER do corpus depois de cada etapa de texto, nas reservadas (estimativa honesta) e em todas as páginas.

In [ ]:
mostrar_html('<h3>WER norm. final (%)</h3>' + summary_html(RESULTADOS, RESERVADAS_N))
for titulo, paginas in (('reservadas', sorted(RESERVADAS_N)), ('todas as páginas', sorted(RESULTADOS[NOME]))):
    etapas = stage_corpus(RESULTADOS[NOME], paginas)
    mostrar_html(f'<h3>WER norm. do corpus por etapa: {titulo} {paginas}</h3>'
                 + bar_svg({rotulo: 100 * s['wer_norm'] for rotulo, s in etapas}))

# textos finais em resultados/textos/pagina_NN.txt
pasta = Path('resultados/textos')
pasta.mkdir(parents=True, exist_ok=True)
for numero, etapas in RESULTADOS[NOME].items():
    (pasta / f'pagina_{numero:02d}.txt').write_text(etapas[-1].text, encoding='utf-8')
print('textos salvos em', pasta)